<a href="https://www.kaggle.com/code/nbanti943gmailcon/prediction-irrigation?scriptVersionId=309595978" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e4/train.csv
/kaggle/input/competitions/playground-series-s6e4/test.csv


In [2]:
#libary load
import pandas as pd
import seaborn as sns
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
import lightgbm as lgb
import xgboost as xgb

# Set random seeds for reproducibility

In [3]:
train=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")
test=pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/test.csv")


In [4]:
train.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [5]:
test.head()

,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
0,630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
1,630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
2,630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
3,630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
4,630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [6]:
from sklearn.preprocessing import LabelEncoder

target = "Irrigation_Need"

# Split
X = train.drop(columns=[target])
y = train[target]

# ✅ Correct: categorical columns select करो
cat_cols = X.select_dtypes(include='object').columns

# Encode categorical features
for col in cat_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))

# Encode target
target_le = LabelEncoder()
y_encoded = target_le.fit_transform(y)

In [7]:
from sklearn.preprocessing import OrdinalEncoder

# categorical columns
cat_cols = [
    'Soil_Type', 'Crop_Type', 'Crop_Growth_Stage',
    'Season', 'Irrigation_Type', 'Water_Source',
    'Mulching_Used', 'Region'
]

# Create encoder
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

# TRAIN encode
X[cat_cols] = encoder.fit_transform(X[cat_cols].astype(str))

# TEST encode (VERY IMPORTANT)
test[cat_cols] = encoder.transform(test[cat_cols].astype(str))

In [8]:
# Setup StratifiedKFold to maintain class distribution ratios
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

# Arrays to store out-of-fold and test predictions
oof_preds_lgb = np.zeros((len(X), 3))
oof_preds_xgb = np.zeros((len(X), 3))
test_preds_lgb = np.zeros((len(test), 3))
test_preds_xgb = np.zeros((len(test), 3))

# LightGBM Parameters (Regularized to prevent overfitting)
lgb_params = {
    'objective': 'multiclass',
    'num_class': 3,
    'learning_rate': 0.03,
    'max_depth': 6,
    'num_leaves': 31,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'class_weight': 'balanced', # CRITICAL: Penalizes errors on minority class
    'n_estimators': 1500,
    'random_state': 42,
    'verbose': -1
}

# XGBoost Parameters
xgb_params = {
    'objective': 'multi:softprob',
    'num_class': 3,
    'learning_rate': 0.03,
    'max_depth': 6,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'n_estimators': 1500,
    'random_state': 42,
    'eval_metric': 'mlogloss',
    'early_stopping_rounds': 100 # <--- MOVED HERE
}

scores_lgb, scores_xgb = [], []

print("Starting Cross-Validation Training...\n")

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_encoded)):
    print(f"--- Fold {fold+1} ---")
    
    X_train, y_train = X.iloc[train_idx], y_encoded[train_idx]
    X_val, y_val = X.iloc[val_idx], y_encoded[val_idx]
    
    # Calculate sample weights for XGBoost (LGBM handles this natively via 'balanced')
    train_weights = compute_sample_weight(class_weight='balanced', y=y_train)
    
    # --- Train LightGBM ---
    model_lgb = lgb.LGBMClassifier(**lgb_params)
    model_lgb.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    # --- Train XGBoost ---
    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(
        X_train, y_train,
        sample_weight=train_weights,
        eval_set=[(X_val, y_val)],
        verbose=False # <--- REMOVED early_stopping_rounds from here
    )
    
    # Store OOF Probability Predictions
    oof_preds_lgb[val_idx] = model_lgb.predict_proba(X_val)
    oof_preds_xgb[val_idx] = model_xgb.predict_proba(X_val)
    
    # Evaluate Fold
    val_pred_class_lgb = np.argmax(oof_preds_lgb[val_idx], axis=1)
    val_pred_class_xgb = np.argmax(oof_preds_xgb[val_idx], axis=1)
    
    fold_score_lgb = balanced_accuracy_score(y_val, val_pred_class_lgb)
    fold_score_xgb = balanced_accuracy_score(y_val, val_pred_class_xgb)
    
    scores_lgb.append(fold_score_lgb)
    scores_xgb.append(fold_score_xgb)
    
    print(f"LGBM Balanced Acc: {fold_score_lgb:.5f} | XGB Balanced Acc: {fold_score_xgb:.5f}")
    
    # Accumulate Test Predictions
    test_preds_lgb += model_lgb.predict_proba(test) / n_splits
    test_preds_xgb += model_xgb.predict_proba(test) / n_splits

print("\n" + "="*40)
print(f"Mean CV LGBM Balanced Acc: {np.mean(scores_lgb):.5f}")
print(f"Mean CV XGB Balanced Acc:  {np.mean(scores_xgb):.5f}")
print("="*40)

Starting Cross-Validation Training...

--- Fold 1 ---
LGBM Balanced Acc: 0.96743 | XGB Balanced Acc: 0.96845
--- Fold 2 ---
LGBM Balanced Acc: 0.96869 | XGB Balanced Acc: 0.96998
--- Fold 3 ---
LGBM Balanced Acc: 0.96970 | XGB Balanced Acc: 0.96998
--- Fold 4 ---
LGBM Balanced Acc: 0.96801 | XGB Balanced Acc: 0.96849
--- Fold 5 ---
LGBM Balanced Acc: 0.96783 | XGB Balanced Acc: 0.96839

Mean CV LGBM Balanced Acc: 0.96833
Mean CV XGB Balanced Acc:  0.96906


In [9]:
submission  = '/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv'

In [10]:
# 1. Load the actual DataFrame from the path
sub_path = '/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv'
submission = pd.read_csv(sub_path)

# 2. Ensemble by averaging predicted probabilities
ensemble_probs = (test_preds_lgb + test_preds_xgb) / 2

# 3. Retrieve final predicted class indices
final_pred_indices = np.argmax(ensemble_probs, axis=1)

# 4. Decode numerical predictions back to original labels
final_labels = target_le.inverse_transform(final_pred_indices)

# 5. Assign to the DataFrame
submission['Irrigation_Need'] = final_labels

# 6. Save to CSV
submission.to_csv('submission.csv', index=False)

print("Success! 'submission.csv' generated.")
submission.head()

Success! 'submission.csv' generated.


,id,Irrigation_Need
0,630000,Low
1,630001,Medium
2,630002,Low
3,630003,Medium
4,630004,Medium
